# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library.

### Dataset Source
The FAIR² dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets in the dataset, along with their field and column `@id` values.

Each entity is **uniquely identified by its `@id`**. Below, record sets and their fields are listed using their `@id`s.

In [ ]:
# List all available record sets
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"RecordSet: {rs['@id']} | Name: {rs.get('name', '[No name]')}")

    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print('  Fields:')
        for field in fields:
            if isinstance(field, dict):
                field_id = field.get('@id', field)
            else:
                field_id = field
            print(f"    - Field @id: {field_id}")

    if 'column' in rs:
        columns = rs['column']
        if isinstance(columns, dict):
            columns = [columns]
        print('  Columns:')
        for col in columns:
            if isinstance(col, dict):
                col_id = col.get('@id', col)
            else:
                col_id = col
            print(f"    - Column @id: {col_id}")
    print()

## 3. Data Extraction
Extract data from available record sets using their `@id`.

For this example, we will load all record sets into Pandas DataFrames.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        # This loads all rows in the record set
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet {record_set_id} with {len(df)} records and columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalization, grouping, etc.

- Select a numeric field for demonstration. 
- Any field, column, or group is referenced by its `@id`.
- Adjust the chosen `record_set_id` and `numeric_field_id` as needed, based on the printed overview above.

In [ ]:
# === Parameters below: Replace with a real @id from previous overview ===
# Example usage: record_set_id = 'cr:ProteinRecords', numeric_field_id = 'cr:MW'

record_set_id = record_set_ids[0] if len(record_set_ids) else None  # Replace if needed
df = dataframes.get(record_set_id)
if df is not None:
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Choose first numeric column
    else:
        numeric_field_id = None
else:
    numeric_field_id = None

if df is not None and numeric_field_id is not None:
    print(f"Analyzing numeric field: {numeric_field_id}")
    # Filter: keep records with value > threshold (arbitrary demonstration)
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field if candidates exist
    other_cols = [col for col in df.columns if col != numeric_field_id]
    group_field_id = other_cols[0] if other_cols else None
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields quickly. For example, plot the histogram of a numeric field, or boxplot grouped by another field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
We explored the Mass Spectrometry Analysis dataset using `mlcroissant`, identified its structure using each entity's `@id`, loaded its data programmatically, and performed quick exploratory analysis. The dataset is ready for advanced analysis and downstream machine learning tasks.

**Be sure to consult the schema and field documentation for precise meaning and unit of each field before using the data for publication or further research.**